## Städning!
- Läs in de olika dataseten
- se vad som behöver städas (kan även se i eda-filerna)
- testa städningen här först innan vi skapar en .py för att generera ny samlad fil
- SPARA DET SOM STÄDAS I "df_cleaned"

### Dataset 1 - artists
- kolla att inget id är tomt - om det är det så kasta
- kolla om name är tomt - om det är det så kasta
- ta bort "followers" och "popularity" - dom behöver vi ej
- "genres"
    - om tom - kasta
    - klumpa ihop "små genrer" till stora populära genrer tex. "pop", "rock" osv
- döp om kolumner: 'artist_id', 'artist_name'

In [4]:
import os
import pandas as pd

df_artists = pd.read_csv('../data/raw/artists.csv')

df_artists.head()

,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0


In [ ]:
# kolla "name"
df_artists["name"].isna().sum()

# Visar hela raderna där den specifika kolumnen är tom
df_artists[df_artists['name'].isna()]

,id,followers,genres,name,popularity
260011,2ZO5sFPyILDQkFdS4HJPFS,8718.0,"['cascadian black metal', 'post-black metal']",NaN,29
417604,4zevbrEdBtYUhAZgCGDtxz,270.0,[],NaN,4
754450,5YVrJKFCv5krBGjhMya576,414.0,[],NaN,5


In [13]:
# Kasta tomma rader i "name" och spara i df_cleaned
# Skapar en ny dataframe utan de tomma raderna i 'kolumnnamn'
df_cleaned = df_artists.dropna(subset=['name'])
df_cleaned.head()

,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0


### 'id' - unika och inga null värden

In [ ]:
# Samma sak med id - kolla om det finns tomma 
df_artists["id"].isna().sum()

# kolla om 'id' är unikt
df_artists['id'].is_unique

# Hitta eventuella dubbletter:
df_artists[df_artists['id'].duplicated(keep=False)]

,id,followers,genres,name,popularity


#### Kasta kolumnerna 'popularity' och 'followers'

In [21]:
relevanta_kolumner = ['id', 'name' ,'genres']
df_cleaned = df_cleaned[relevanta_kolumner]
df_cleaned.head()

,id,name,genres
0,22mLrN5fkppmuUPsHx6i2G,Vera Dulova,[classical]
1,1OsJZxSshQD4BCg1VtwxsN,Anna Larsson,[classical]
2,3zQdpHMTdJnV4aCzGqCBYK,Javad Maroufi,[pop]
3,7frYUe4C7A42uZqCzD34Y4,Sultaan,"[hip hop, pop]"
4,6acbdy69rtlv8m9EW31MYl,Phyno,"[hip hop, pop]"


### Gruppera genres

In [ ]:
import ast
import re
import pandas as pd

MAP = {
    "pop": ["pop"],
    "rock": ["rock"],
    "hip hop": ["hip hop", "rap", "drill"],
    "electronic": ["house", "edm", "electro", "rawstyle"],
    "latin": [
        "latin",
        "mexican",
        "corrido",
        "ranchera",
        "norteno",
        "banda",
        "sierreno",
        "cumbia",
    ],
    "brazilian": ["mpb", "pagode", "bossa nova", "forro", "samba", "axe"],
    "soundtrack": [
        "soundtrack",
        "show tunes",
        "movie tunes",
        "hollywood",
        "broadway",
        "scorecore",
    ],
    "jazz": ["jazz"],
    "metal": ["metal"],
    "country": ["country"],
    "r&b": ["r&b"],
    "soul": ["soul"],
    "folk": ["folk"],
    "indie": ["indie"],
    "punk": ["punk"],
    "reggae": ["reggae"],
    "blues": ["blues"],
    "classical": ["classical"],
    "funk": ["funk"],
}

# 1. Bygg regex per huvudkategori (skiftlägesokänslig)
rx = {
    cat: re.compile(
        "|".join(re.escape(k) for k in keywords), flags=re.IGNORECASE
    )
    for cat, keywords in MAP.items()
}


# 2. Säker tolkning av sträng-listan
def safe_eval(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


# 3. Mappa varje genre-lista till dina övergripande kategorier
def group(genres_list):
    matched_categories = set()
    for g in genres_list:
        g_clean = str(g).strip()
        for cat, pattern in rx.items():
            if pattern.search(g_clean):
                matched_categories.add(cat)
    return sorted(matched_categories)


# 4. Kör på ditt dataset och spara i df_cleaned
df_cleaned = df_artists.copy()
df_cleaned["genres"] = df_cleaned["genres"].map(safe_eval).map(group)

# 5. Filtrera bort rader som blev tomma (dvs. inte matchade någon kategori)
df_cleaned = df_cleaned[df_cleaned["genres"].str.len() > 0].reset_index(
    drop=True
)

df_cleaned.head(20)

,id,followers,genres,name,popularity
0,22mLrN5fkppmuUPsHx6i2G,59.0,[classical],Vera Dulova,3
1,1OsJZxSshQD4BCg1VtwxsN,155.0,[classical],Anna Larsson,19
2,3zQdpHMTdJnV4aCzGqCBYK,3918.0,[pop],Javad Maroufi,26
3,7frYUe4C7A42uZqCzD34Y4,53636.0,"[hip hop, pop]",Sultaan,53
4,6acbdy69rtlv8m9EW31MYl,72684.0,"[hip hop, pop]",Phyno,51
5,72578usTM6Cj5qWsi471Nc,248568.0,"[folk, pop, rock]",Raghu Dixit,52
6,7b6Ui7JVaBDEfZB9k6nHL0,701766.0,"[indie, pop, rock]",The Local Train,57
7,1lNpPgcIPULhVgDxvI7xLs,4196.0,[electronic],Monoplay,32
8,0YdFvpH7MvXv5vBsfGvt7g,1009.0,"[electronic, pop]",Alice Berg,41
9,4bvGDTEPFnllKiJaEZGuXk,6395.0,[folk],Achint,49


In [ ]:
# Kolla att det inte finns kvar några tomma rader
df_cleaned['genres'].isna().sum()

np.int64(0)

In [24]:
# Byt namn på 'id' och 'name'
df_cleaned = df_cleaned.rename(columns={'name': 'artist_name', 'id': 'artist_id'})
df_cleaned.head()

,artist_id,followers,genres,artist_name,popularity
0,22mLrN5fkppmuUPsHx6i2G,59.0,[classical],Vera Dulova,3
1,1OsJZxSshQD4BCg1VtwxsN,155.0,[classical],Anna Larsson,19
2,3zQdpHMTdJnV4aCzGqCBYK,3918.0,[pop],Javad Maroufi,26
3,7frYUe4C7A42uZqCzD34Y4,53636.0,"[hip hop, pop]",Sultaan,53
4,6acbdy69rtlv8m9EW31MYl,72684.0,"[hip hop, pop]",Phyno,51


### Dataset 2 - Tracks
- kasta onödiga kolumner 
- kolla så att inte 'id' eller 'name' saknar värden - kasta tomma
- döp om kolumner
- 'relese_date' - endast årtal 

In [3]:
df_tracks = pd.read_csv('../data/raw/tracks.csv')
df_tracks.head()

,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.4450,0,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.2630,0,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.1770,1,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.321,0.0946,7,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922,0.402,0.1580,3,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4


In [29]:
# Kolla tomma väden på 'id' och 'name'
df_tracks[["id", "name"]].isna().sum()

# Kasta rader där 'id' eller 'name' saknas
df_tracks = df_tracks.dropna(subset=["id", "name"])

df_tracks.head()


,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.4450,0,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.2630,0,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.1770,1,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.321,0.0946,7,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922,0.402,0.1580,3,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4


### 2. 'release_date' - behåll endast årtal


In [34]:
# Datumen är ofta blandade (t.ex. '1994-05-12' eller '1994').
# Vi tar de första 4 tecknen och omvandlar till heltal (int):
df_tracks["release_year"] = (
    df_tracks["release_year"].astype(str).str[:4].astype(int)
)

df_tracks.head()
#df_tracks["release_date"].isna().sum()

,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,release_year
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.4450,...,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3,1922
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.2630,...,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1,1922
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.1770,...,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5,1922
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.321,0.0946,...,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3,1922
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922,0.402,0.1580,...,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4,1922


### Spara endast relevanta kolumner

In [35]:
relevanta_kolumner = ['id', 'name' ,'artists', 'id_artists', "release_year"]
df_tracks = df_tracks[relevanta_kolumner]
df_tracks.head()

,id,name,artists,id_artists,release_year
0,35iwgR4jXetI318WEWsa1Q,Carve,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922


In [37]:
# byt namn på kolumnerna
df_tracks = df_tracks.rename(columns={'name': 'track_name', 'id': 'track_id', 'artists': 'artist_name', 'id_artists': 'artist_id'})
df_tracks.head()

,track_id,track_name,artist_name,artist_id,release_year
0,35iwgR4jXetI318WEWsa1Q,Carve,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922


### Dataset 3 - dataset (för album)
- 'track_id' - ta bort dubletter (viktigt för join)
- 'album_name' - lägg till 'missing album' för tomma rader
- kasta resten av kolumnerna (vi behöver dom ej)

In [5]:
df_dataset = pd.read_csv('../data/raw/dataset.csv')
df_dataset.head()


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [39]:
import pandas as pd

# 1. Läs in (eller välj ut) endast relevanta kolumner
df_albums = df_dataset[['track_id', 'album_name']].copy()

# 2. Ta bort dubbletter på track_id (kritiskt så inte joinen exploderar i rader)
df_albums = df_albums.drop_duplicates(subset=['track_id']).reset_index(drop=True)

# 3. Fyll eventuella tomma albumfält i filen
df_albums['album_name'] = df_albums['album_name'].fillna('missing album')

df_albums.head()
df_albums['track_id'].count()

np.int64(89741)